# Multi-Strategy Quantitative Portfolio Optimization Engine
### Operations Research for Quantitative Finance | Convex Quadratic Programming & Linear Programming (Min CVaR)

This notebook demonstrates institutional-grade portfolio optimization strategies across 5 years of daily market histories (AAPL, MSFT, JPM, XOM, JNJ, TLT):
1. **Markowitz Mean-Variance Optimization (Max Sharpe):** Quadratic programming maximizing risk-adjusted excess returns.
2. **Equal Risk Parity (ERC):** Non-linear optimization equalizing marginal risk contributions across asset classes.
3. **PMPT Maximum Sortino Ratio:** Downside semi-variance optimization penalizing only negative return volatility.
4. **Tail-Risk Minimization (Min CVaR / 95% Expected Shortfall):** Convex linear optimization minimizing severe left-tail losses under non-normal asset return distributions.

In [1]:
import os
import sys
import numpy as np
import pandas as pd

# Add current directory to path
sys.path.insert(0, os.getcwd())

from src.data_loader import PortfolioDataLoader
from src.portfolio_optimizer import MultiStrategyPortfolioOptimizer

# 1. Ingest 5-Year Multi-Asset Price Series
loader = PortfolioDataLoader(data_dir="data")
data = loader.load_price_data()

print(f"Multi-Asset Universe: {data['tickers']}")
print(f"Historical Trading Days: {len(data['daily_returns']):,} days")
print(f"Ledoit-Wolf Regularized Covariance Matrix Shape: {data['shrunk_covariance'].shape}")

Multi-Asset Universe: ['AAPL', 'MSFT', 'JPM', 'XOM', 'JNJ', 'TLT']
Historical Trading Days: 1,259 days
Ledoit-Wolf Regularized Covariance Matrix Shape: (6, 6)


## 2. Execute Multi-Strategy Optimization (Markowitz, Risk Parity, Sortino, Min CVaR)

In [3]:
optimizer = MultiStrategyPortfolioOptimizer(data, risk_free_rate=0.04)

sharpe_res = optimizer.optimize_max_sharpe()
erc_res = optimizer.optimize_risk_parity()
sortino_res = optimizer.optimize_max_sortino()
cvar_res = optimizer.optimize_min_cvar(alpha=0.95)

print("=" * 95)
print(f"{'Strategy':<30} | {'Exp Return':<12} | {'Volatility':<12} | {'Sharpe':<10} | {'Sortino':<10}")
print("-" * 95)
print(f"{'Markowitz Max Sharpe':<30} | {sharpe_res['expected_annual_return']*100:<11.2f}% | {sharpe_res['annual_volatility']*100:<11.2f}% | {sharpe_res['sharpe_ratio']:<10.2f} | -")
print(f"{'Equal Risk Parity (ERC)':<30} | {erc_res['expected_annual_return']*100:<11.2f}% | {erc_res['annual_volatility']*100:<11.2f}% | {erc_res['sharpe_ratio']:<10.2f} | -")
print(f"{'PMPT Max Sortino':<30} | {sortino_res['expected_annual_return']*100:<11.2f}% | {sortino_res['annual_volatility']*100:<11.2f}% | {'-':<10} | {sortino_res['sortino_ratio']:<10.2f}")
print(f"{'Tail Risk Min CVaR (95% ES)':<30} | {cvar_res['expected_annual_return']*100:<11.2f}% | {cvar_res['annual_volatility']*100:<11.2f}% | {'-':<10} | -")
print("=" * 95)

print("\nOptimal Markowitz Asset Allocation Weights:")
for t, w in sharpe_res['weights'].items():
    print(f"  * {t:<5}: {w*100:6.2f}%")

# Discrete $100k allocation
discrete = optimizer.compute_discrete_allocation(sharpe_res['weights'], total_capital=100000.0)
print(f"\nDiscrete $100,000 Portfolio Execution:")
for t, shares in discrete['allocated_shares'].items():
    print(f"  * Buy {shares:4d} shares of {t}")
print(f"  * Total Capital Allocated: ${discrete['allocated_capital']:,.2f} | Uninvested Cash: ${discrete['leftover_cash']:.2f}")

Strategy                       | Exp Return   | Volatility   | Sharpe     | Sortino   
-----------------------------------------------------------------------------------------------
Markowitz Max Sharpe           | 19.36      % | 16.83      % | 0.91       | -
Equal Risk Parity (ERC)        | 6.02       % | 8.82       % | 0.23       | -
PMPT Max Sortino               | 19.60      % | 17.10      % | -          | 0.93      
Tail Risk Min CVaR (95% ES)    | 2.68       % | 8.20       % | -          | -

Optimal Markowitz Asset Allocation Weights:
  * AAPL :  17.08%
  * MSFT :   0.00%
  * JPM  :  81.92%
  * XOM  :   1.00%
  * JNJ  :   0.00%
  * TLT  :   0.00%

Discrete $100,000 Portfolio Execution:
  * Buy   64 shares of AAPL
  * Buy    0 shares of MSFT
  * Buy  229 shares of JPM
  * Buy    6 shares of XOM
  * Buy    0 shares of JNJ
  * Buy    0 shares of TLT
  * Total Capital Allocated: $99,883.16 | Uninvested Cash: $116.84
